# Product Dimensions BYOC Example

This notebook demonstrates the BYOC package flow for **Product Dimensions**: request a short-lived CodeArtifact credential from the Bria Engine, install **`product_dimensions`** from the **`bria-product-dimensions`** repository, then run the local `ProductDimensions` pipeline on product images.

Product Dimensions renders marketplace-ready dimension callouts from product photos. It reuses background removal when the input image does not already have transparency, then draws dimension lines, optional title, weight, and capacity text.

**Prerequisites**

- `BRIA_API_TOKEN` in the environment.
- `HF_TOKEN` in the environment with access to `briaai/RMBG-2.0`.
- Python 3.10 or newer.
- Network access to Bria Engine, AWS CodeArtifact, and Hugging Face.
- Runtime support for the underlying `remove_bg` model package.

## 1. CodeArtifact Token

Request a short-lived credential for the **`bria-product-dimensions`** PyPI repository. The credential is used as the CodeArtifact password with username `aws`.

In [ ]:
import json
import os
from urllib.error import HTTPError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN")
if not BRIA_API_TOKEN:
    raise RuntimeError("Set BRIA_API_TOKEN in the environment before running this notebook.")

repository = "bria-product-dimensions"
params = urlencode({"repository": repository})
url = f"https://engine.prod.bria-api.com/v2/auth/access/code_artifact?{params}"
request = Request(
    url,
    headers={
        "api_token": BRIA_API_TOKEN,
        "token": BRIA_API_TOKEN,
        "User-Agent": "BriaPlatform/APIdocs/LLMsAgent",
    },
)

try:
    with urlopen(request, timeout=60) as response:
        payload = json.loads(response.read().decode("utf-8"))
except HTTPError as exc:
    error_body = exc.read().decode("utf-8", errors="replace")
    try:
        error_payload = json.dumps(json.loads(error_body), indent=2)
    except json.JSONDecodeError:
        error_payload = error_body or "<empty response body>"
    raise RuntimeError(
        f"CodeArtifact token request failed for repository {repository!r} "
        f"with HTTP {exc.code}: {exc.reason}\n{error_payload}"
    ) from exc

result = payload.get("result") or {}
CODE_ARTIFACT_PASSWORD = (result.get("authorization_token") or "").strip()
if not CODE_ARTIFACT_PASSWORD:
    raise RuntimeError(f"CodeArtifact response did not include an authorization token: {payload}")

expiration = result.get("expiration")
print("CodeArtifact credential acquired." + (f" Expires: {expiration}" if expiration else ""))

## 2. Install dependencies

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote

# Pip imports call os.getcwd(). If Jupyter started from a folder that was later removed, inherited cwd breaks pip.
try:
    pip_cwd = str(Path.cwd())
except FileNotFoundError:
    pip_cwd = str(Path.home())
    os.chdir(pip_cwd)

encoded_password = quote(CODE_ARTIFACT_PASSWORD, safe="")
codeartifact_base = (
    "https://aws:"
    + encoded_password
    + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi"
)
product_dimensions_index = codeartifact_base + "/bria-product-dimensions/simple/"
remove_bg_index = codeartifact_base + "/bria-remove-bg/simple/"
external_index = codeartifact_base + "/bria-external/simple/"

pytorch_index = "https://download.pytorch.org/whl/cu126"


def run_pip_install(install_cmd):
    redacted_cmd = " ".join(part.replace(encoded_password, "<redacted>") for part in install_cmd)
    print("Running:", redacted_cmd, flush=True)
    process = subprocess.Popen(
        install_cmd,
        cwd=pip_cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line.replace(encoded_password, "<redacted>"), end="", flush=True)

    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError("pip install failed. See pip output above; authenticated index URLs were redacted.")


run_pip_install(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--force-reinstall",
        "torch",
        "torchvision",
        "--index-url",
        pytorch_index,
    ]
)

run_pip_install(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        "product_dimensions",
        "--extra-index-url",
        product_dimensions_index
    ]
)

## 3. Start The Local Pipeline

The example disables model compilation for a simpler first run. Tune the nested `rmbg_config` for your deployment environment.

If the install cell changed core runtime packages, restart the notebook kernel before running this setup cell so Python imports the installed versions cleanly.

In [ ]:
import logging
from pathlib import Path

from product_dimensions import ProductDimensions, ProductDimensionsConfig

logging.basicConfig(level=logging.INFO)

outputs_dir = Path("outputs")
outputs_dir.mkdir(exist_ok=True)

pipeline = ProductDimensions(config=ProductDimensionsConfig(rmbg_config={"compile_model": False}))
pipeline.setup()
print("Product Dimensions pipeline setup complete.")

## 4. Render A Dimension Image

Pass either a URL or a local image path into `ProductDimensionsInput.image`. Provide at least one `DimensionEntry`. Duplicate entries with the same name and position are merged into dual-unit labels when `units_display="dual_parens"`.

In [ ]:
import time

from IPython.display import display
from product_dimensions import (
    CapacityInput,
    DimensionEntry,
    ProductDimensionsInput,
    WeightInput,
)
from product_dimensions.schemas import CapacityUnit, DimensionName, LengthUnit, OutputFormat, Position, WeightUnit

product_image = "https://bria-test-images.s3.us-east-1.amazonaws.com/strawberry.jpg"

task = ProductDimensionsInput(
    image=product_image,
    title="Fresh Strawberry",
    style="default",
    units_display="dual_parens",
    background="white",
    dimensions=[
        DimensionEntry(name=DimensionName.HEIGHT, value=7.5, unit=LengthUnit.CM, position=Position.LEFT),
        DimensionEntry(name=DimensionName.HEIGHT, value=2.95, unit=LengthUnit.IN, position=Position.LEFT),
        DimensionEntry(name=DimensionName.WIDTH_BOTTOM, value=5.2, unit=LengthUnit.CM, position=Position.BOTTOM),
        DimensionEntry(name=DimensionName.WIDTH_BOTTOM, value=2.05, unit=LengthUnit.IN, position=Position.BOTTOM),
    ],
    weight=WeightInput(value=18, unit=WeightUnit.G),
    capacity=CapacityInput(value=1, unit=CapacityUnit.QT),
    output_format=OutputFormat.PNG,
    output_size=1400,
)

t0 = time.time()
result = pipeline.execute(task)
output_path = outputs_dir / "strawberry_product_dimensions.png"
result.image.save(output_path)

print(f"Generated {result.image.size} dimension image in {time.time() - t0:.1f}s")
print(f"Saved to {output_path}")
display(result.image)

## 5. Cleanup

Release resources held by the underlying pipeline when you are done.

In [ ]:
pipeline.cleanup()
print("Cleanup done.")